[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/othnielObasi/airg-cookbook/blob/main/use_cases/21_surge_receipt_verification.ipynb)

# 🔐 SURGE Receipt Verification — Cryptographic Compliance Audit

Every AIRG governance decision produces an Ed25519-signed receipt linked in a hash chain.
This notebook walks through verifying individual receipts, validating chain integrity,
and exporting compliance bundles for SOC2/ISO 27001 audits.

**What you'll learn:**
- How SURGE receipts provide tamper-proof governance logs
- Verify individual receipt signatures and digests
- Validate the entire hash chain for integrity
- Export compliance bundles for auditors

**Setup:** Set `GOVERNOR_API_KEY` in Colab Secrets (🔑 icon) or as an environment variable.

## 1. Install & Configure

In [ ]:
!pip install -q httpx

import os, httpx

try:
    from google.colab import userdata
    API_KEY = userdata.get("GOVERNOR_API_KEY")
except Exception:
    API_KEY = os.getenv("GOVERNOR_API_KEY", "YOUR_KEY")

BASE = os.getenv("GOVERNOR_URL", "https://api.airg.nov-tia.com")
headers = {"X-API-Key": API_KEY}
print(f"Connected to {BASE}")

## 2. Generate Some Governance Decisions

In [ ]:
# Create a few evaluations to ensure we have receipts
test_actions = [
    ("audit-agent", "read_file", {"path": "/data/report.csv"}),
    ("audit-agent", "shell_exec", {"command": "rm -rf /tmp/old"}),
    ("audit-agent", "http_request", {"url": "https://api.internal/data"}),
]

print("Generating governance decisions...\n")
for agent, tool, args in test_actions:
    r = httpx.post(f"{BASE}/actions/evaluate", headers=headers, json={
        "agent_id": agent, "tool": tool, "args": args,
    })
    d = r.json()
    print(f"  {tool:15s} -> {d['decision']}  risk={d['risk_score']}")

print("\nReceipts generated for each decision.")

## 3. Check SURGE Chain Status

In [ ]:
status = httpx.get(f"{BASE}/surge/v2/status", headers=headers).json()

print("SURGE V2 Chain Status")
print("-" * 40)
print(f"  Chain length     : {status['chain_length']}")
print(f"  Signing algorithm: {status.get('signing_algorithm', 'Ed25519')}")
print(f"  Signing enabled  : {status.get('signing_enabled', False)}")
print(f"  Chain intact     : {status.get('chain_intact', 'unknown')}")


## 4. Verify Chain Integrity

In [ ]:
verify = httpx.get(f"{BASE}/surge/v2/verify", headers=headers).json()

intact = verify["valid"]
icon = "VALID" if intact else "BROKEN"
print(f"Chain Integrity: [{icon}]")
print(f"  Receipts checked : {verify['receipts_checked']}")
print(f"  First broken at  : {verify.get('first_broken_at', 'None')}")
print(f"  Errors           : {len(verify.get('errors', []))}")
print(f"  Warnings         : {len(verify.get('warnings', []))}")

if intact:
    print("\n  The entire hash chain is cryptographically valid.")
    print("  No receipts have been tampered with or deleted.")
else:
    print("\n  WARNING: Chain integrity compromised!")
    print("  Some receipts may have been tampered with.")


## 5. Verify a Single Receipt

In [ ]:
receipts = httpx.get(f"{BASE}/surge/v2/receipts?limit=3",
                     headers=headers).json()

if receipts:
    for receipt in receipts[:3]:
        rid = receipt["receipt_id"]
        rv = httpx.get(f"{BASE}/surge/v2/receipts/{rid}/verify",
                       headers=headers).json()
        print(f"Receipt {rid}:")
        print(f"  Digest valid    : {rv['digest_valid']}")
        print(f"  Chain link valid : {rv['chain_link_valid']}")
        print(f"  Signature valid  : {rv['signature_valid']}")
        compliance = receipt.get("compliance", {})
        print(f"  Compliance       : {list(compliance.keys()) if compliance else 'none'}")
        print()
else:
    print("No receipts found yet.")


## 6. Export Compliance Bundle

In [ ]:
export = httpx.get(f"{BASE}/surge/v2/export", headers=headers).json()

print("Compliance Export Bundle")
print("-" * 40)
print(f"  Total receipts: {export['total_receipts']}")
print(f"  Block rate    : {export['summary']['block_rate_pct']}%")
print(f"  Export size   : ~{len(str(export)) // 1024}KB JSON")
print()
print("  This bundle contains:")
print("  - Every governance decision with full context")
print("  - Ed25519 signatures for each receipt")
print("  - Hash chain links for tamper detection")
print("  - Summary statistics for auditor review")
print()
print("  Ready for SOC2 / ISO 27001 / HIPAA compliance review.")

## Summary

| Feature | What It Proves |
|---|---|
| **Ed25519 Signatures** | Each receipt is cryptographically signed |
| **Hash Chain** | Receipts are linked; tampering breaks the chain |
| **Single Verify** | Verify any individual decision |
| **Chain Verify** | Validate the entire audit log at once |
| **Export** | One-click compliance bundle for auditors |

→ [Full API docs](https://api.airg.nov-tia.com/docs) · [More recipes](https://github.com/othnielObasi/airg-cookbook)